# Motor de Web Scraping — Ofertas Financieras Competencia Automoción
**Honda Financial Services | TFM BSM Barcelona**

Extrae ofertas de financiación (TIN, TAE, comisión de apertura, cuota, plazo, etc.) de webs de competidores usando scraping + LLM (Claude).

## 1. Instalación de dependencias (ejecutar solo en Colab)

In [ ]:
# Ejecuta esta celda la primera vez en Google Colab (tarda ~1 minuto)
!pip install -q requests beautifulsoup4 selenium pandas webdriver-manager openai
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y -q ./google-chrome-stable_current_amd64.deb
print("Dependencias instaladas correctamente")

## 2. Imports y configuración

In [ ]:
import requests
import time
import json
import re
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from openai import OpenAI

print("Librerías cargadas correctamente")

In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CAMPOS_OFERTA = [
    "marca",
    "modelo",
    "precio_vehiculo",
    "cuota_mensual",
    "plazo_meses",
    "entrada",
    "tin",
    "tae",
    "comision_apertura",
    "valor_residual",
    "importe_financiado",
    "tipo_financiacion",
    "fecha_fin_oferta",
    "url",
    "fecha_extraccion"
]

print(f"API Key OpenAI cargada: {'OK' if OPENAI_API_KEY else 'ERROR — revisa los Secrets'}")

## 3. Funciones de scraping

In [ ]:
def scrape_estatico(url, reintentos=3, pausa=2):
    for intento in range(reintentos):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            print(f"  [intento {intento+1}/{reintentos}] Error en {url}: {e}")
            time.sleep(pausa * (intento + 1))
    return None


def crear_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")
    driver_path = ChromeDriverManager().install()
    return webdriver.Chrome(service=Service(driver_path), options=options)


def scroll_hasta_el_final(driver, pausas=8):
    for i in range(pausas):
        driver.execute_script("window.scrollBy(0, document.body.scrollHeight);")
        time.sleep(1.5)
    driver.execute_script("window.scrollTo(0, 0);")


def scrape_dinamico(url, espera_extra=3, scroll=False):
    driver = crear_driver()
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(espera_extra)
        if scroll:
            scroll_hasta_el_final(driver)
            time.sleep(2)
        return driver.page_source
    except Exception as e:
        print(f"  Error Selenium en {url}: {e}")
        return None
    finally:
        driver.quit()


def html_a_texto(html, seccion_especial=None, max_chars_legal=4000,
                 pagina_listado=False, umbral_tin=6000):
    """
    Extrae el fragmento relevante del HTML para enviarlo al LLM.

    pagina_listado=True: extrae desde el primer TIN hasta el FINAL del texto,
      sin límite de caracteres, para capturar todas las ofertas del listado.
    umbral_tin: solo aplica cuando pagina_listado=False. TINs más allá de esta
      posición se consideran carrusel y se ignoran.
    """
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "noscript"]):
        tag.decompose()
    texto = soup.get_text(separator=" ", strip=True)
    texto = re.sub(r'\s+', ' ', texto)

    cabecera = texto[:1500]

    # Sección especial (ej. Renault)
    if seccion_especial:
        idx = texto.find(seccion_especial)
        if idx != -1:
            print(f"  Sección '{seccion_especial[:40]}' encontrada en pos {idx}")
            return cabecera + " [...] " + texto[idx:idx + max_chars_legal]
        else:
            print(f"  AVISO: Sección especial no encontrada, usando fallback")

    # Buscar primer TIN
    pos = -1
    for kw in ["TIN:", "TIN :", "T.I.N"]:
        p = texto.find(kw)
        if p != -1:
            pos = p
            break

    if pos != -1:
        if pagina_listado:
            # Modo listado: sin límite, todo el texto desde el primer TIN hasta el final
            bloque = texto[max(0, pos - 200):]
            print(f"  [LISTADO] TIN en pos {pos} — extrayendo {len(bloque)} chars hasta el final")
            return cabecera + " [...] " + bloque
        elif pos < umbral_tin:
            inicio = max(0, pos - 500)
            bloque = texto[inicio:inicio + max_chars_legal]
            print(f"  Texto enviado al LLM: {len(cabecera) + len(bloque)} chars (TIN en pos {pos})")
            return cabecera + " [...] " + bloque
        else:
            print(f"  TIN encontrado en pos {pos} (carrusel, ignorado) — usando cabecera")

    print(f"  Texto enviado al LLM: {len(cabecera)} chars (sin texto legal propio)")
    return cabecera


print("Funciones de scraping definidas")

## 4. Extracción con LLM (Claude)

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

PROMPT_SISTEMA = """Eres un experto en análisis de ofertas de financiación de automóviles en España.
Tu tarea es extraer información estructurada de textos de páginas web de concesionarios.
Devuelve SIEMPRE un JSON válido con los campos indicados.
Si un campo no aparece en el texto, devuelve null para ese campo.
No inventes datos. Solo extrae lo que esté explícitamente en el texto."""


def extraer_oferta_con_llm(texto, marca, url, filtro_producto=None, pagina_listado=False):
    slug = url.rstrip("/").split("/")[-1]
    pista_modelo = (slug.replace("-easy-plus", "").replace("-easy-renting", "")
                    .replace("-easy", "").replace("-", " ").title())

    if filtro_producto:
        instruccion_filtro = (
            f"IMPORTANTE: Extrae ÚNICAMENTE la oferta principal del producto '{filtro_producto}' "
            f"para el modelo '{pista_modelo}'. Devuelve exactamente UNA oferta. "
            f"Ignora cualquier otro modelo o producto mencionado en el texto."
        )
        max_tokens = 1500
    elif pagina_listado:
        instruccion_filtro = (
            "Extrae TODAS las ofertas de financiación que encuentres en el texto legal. "
            "Puede haber múltiples modelos, cada uno con su propia oferta. "
            "Devuelve una entrada por cada modelo distinto con TIN/TAE propio."
        )
        max_tokens = 8000
    else:
        instruccion_filtro = "Extrae la oferta de financiación principal que encuentres."
        max_tokens = 1500

    prompt = f"""{instruccion_filtro}

Analiza el siguiente texto de la web de {marca} ({url}).

Para cada oferta devuelve un objeto JSON con estos campos:
- modelo: nombre comercial del modelo de coche
- tipo_combustible: "gasolina", "diésel", "híbrido", "híbrido enchufable", "eléctrico" o null
- precio_vehiculo: PVP al contado en € (número). Busca "PVP al contado", "precio al contado" o "PVP recomendado"
- cuota_mensual: cuota mensual en € (número)
- plazo_meses: duración en meses (número)
- entrada: entrada inicial en € (número, 0 si no hay)
- tin: TIN en % (número)
- tae: TAE en % (número)
- comision_apertura: importe de la comisión de apertura en € (número, 0 si es gratuita)
- porcentaje_comision_apertura: comisión de apertura en % sobre el capital (número o null)
- valor_residual: última cuota o valor residual en € (número o null)
- importe_financiado: capital total financiado en € (número)
- tipo_financiacion: nombre exacto del producto financiero
- fecha_fin_oferta: fecha límite en YYYY-MM-DD (string o null)

Devuelve SOLO este JSON:
{{"ofertas": [ {{...}} ]}}

TEXTO:
{texto}"""

    try:
        respuesta = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=max_tokens,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": PROMPT_SISTEMA},
                {"role": "user", "content": prompt}
            ]
        )
        datos = json.loads(respuesta.choices[0].message.content)
        ofertas = [o for o in datos.get("ofertas", []) if o is not None]
        for oferta in ofertas:
            oferta["marca"] = marca
            oferta["url"] = url
            oferta["fecha_extraccion"] = str(date.today())
        return ofertas
    except Exception as e:
        print(f"  Error LLM para {url}: {e}")
        return []


print("Cliente OpenAI (gpt-4o-mini) configurado")

## 5. Pipeline completo: scraping + extracción LLM

In [ ]:
def procesar_url(url, marca, usar_selenium=False, scroll=False,
                 seccion_especial=None, filtro_producto=None,
                 pagina_listado=False, max_chars_legal=4000):
    print(f"Procesando: {marca} — {url}")
    html = scrape_dinamico(url, scroll=scroll) if usar_selenium else scrape_estatico(url)
    if not html:
        print(f"  No se pudo descargar {url}")
        return []
    texto = html_a_texto(
        html,
        seccion_especial=seccion_especial,
        max_chars_legal=max_chars_legal,
        pagina_listado=pagina_listado
    )
    ofertas = extraer_oferta_con_llm(
        texto, marca, url,
        filtro_producto=filtro_producto,
        pagina_listado=pagina_listado
    )
    print(f"  Ofertas encontradas: {len(ofertas)}")
    return ofertas


def descubrir_urls_toyota():
    """Extrae desde toyota.es/promociones todas las URLs con 'easy' sin 'renting'."""
    BASE = "https://www.toyota.es"
    print(f"Descubriendo URLs Toyota Easy desde {BASE}/promociones ...")
    html = scrape_estatico(f"{BASE}/promociones")
    if not html:
        print("  No se pudo descargar la página de promociones de Toyota")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    EXCLUIR = ["/promociones/toyota-easy-plus", "/promociones/toyota-easy",
               "/promociones/toyota-easy-complet"]
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ("/promociones/" in href
                and "easy" in href
                and "renting" not in href
                and not any(href.endswith(ex.split("/")[-1]) and "/finance-insurance/" not in href
                            for ex in EXCLUIR)
                and "/finance-insurance/" not in href):
            url_completa = href if href.startswith("http") else BASE + href
            if url_completa not in urls:
                urls.append(url_completa)
    print(f"  URLs Easy encontradas: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


print("Pipeline y auto-descubrimiento Toyota definidos")

## 6. URLs de la competencia

In [ ]:
COMPETENCIA = {
    "TOYOTA": {
        "selenium": False, "scroll": False,
        "seccion_especial": None, "filtro_producto": "Easy Plus",
        "pagina_listado": False, "max_chars_legal": 4000,
        "urls": []  # se auto-descubren desde toyota.es/promociones
    },
    "VOLKSWAGEN": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.volkswagen.es/es/ofertas.html"]
    },
    "PEUGEOT": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.peugeot.es/comprar/ofertas-del-momento.html"]
    },
    "RENAULT": {
        "selenium": False, "scroll": False,
        "seccion_especial": "CONDICIONES LEGALES PARA PENÍNSULA Y BALEARES",
        "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 4000,
        "urls": [
            "https://promociones.renault.es/particulares/clio/",
            "https://promociones.renault.es/particulares/captur/",
            "https://promociones.renault.es/particulares/symbioz/",
            "https://promociones.renault.es/particulares/symbioz-glp/",
            "https://promociones.renault.es/particulares/austral/",
            "https://promociones.renault.es/particulares/arkana/",
            "https://promociones.renault.es/particulares/espace/",
            "https://promociones.renault.es/particulares/rafale/",
            "https://promociones.renault.es/particulares/rafale-phev/"
        ]
    },
    "NISSAN": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.nissan.es/vehiculos/ofertas.html"]
    },
    "HYUNDAI": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 4000,
        "urls": [
            "https://www.hyundai.com/es/es/modelos/kona.html",
            "https://www.hyundai.com/es/es/modelos/tucson.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-bayon.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-i10.html",
            "https://www.hyundai.com/es/es/modelos/i20.html",
            "https://www.hyundai.com/es/es/modelos/i30.html",
            "https://www.hyundai.com/es/es/modelos/i30-fastback.html",
            "https://www.hyundai.com/es/es/modelos/i30wagon.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-santafe-hev.html",
            "https://www.hyundai.com/es/es/modelos/kona-hibrido.html",
            "https://www.hyundai.com/es/es/modelos/tucson-hibrido.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-santafe-phev.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-tucson-phev.html",
            "https://www.hyundai.com/es/es/modelos/inster.html",
            "https://www.hyundai.com/es/es/modelos/kona-electrico.html",
            "https://www.hyundai.com/es/es/modelos/ioniq5.html",
            "https://www.hyundai.com/es/es/modelos/ioniq6.html",
            "https://www.hyundai.com/es/es/modelos/ioniq9.html"
        ]
    },
    "AUDI": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.audi.es/es/compra/promociones/"]
    }
}

print(f"Configuradas {len(COMPETENCIA)} marcas")

## 7. Ejecución del scraping

In [ ]:
# Para probar solo una marca: MARCAS_A_EJECUTAR = ["VOLKSWAGEN"]
MARCAS_A_EJECUTAR = list(COMPETENCIA.keys())

todas_las_ofertas = []

for marca in MARCAS_A_EJECUTAR:
    config = COMPETENCIA[marca]
    print(f"\n{'='*50}\nMARCA: {marca}\n{'='*50}")

    # Toyota: auto-descubrimiento desde toyota.es/promociones
    if marca == "TOYOTA":
        urls = descubrir_urls_toyota()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    else:
        urls = config["urls"]

    if not urls:
        print(f"  Sin URLs para {marca}, saltando.")
        continue

    for url in urls:
        ofertas = procesar_url(
            url, marca,
            usar_selenium=config["selenium"],
            scroll=config.get("scroll", False),
            seccion_especial=config.get("seccion_especial"),
            filtro_producto=config.get("filtro_producto"),
            pagina_listado=config.get("pagina_listado", False),
            max_chars_legal=config.get("max_chars_legal", 4000)
        )
        todas_las_ofertas.extend(ofertas)
        time.sleep(2)

print(f"\n{'='*50}")
print(f"RESUMEN: {len(todas_las_ofertas)} ofertas brutas extraídas")
marcas_con_datos = set(o["marca"] for o in todas_las_ofertas)
print(f"Marcas CON datos: {marcas_con_datos}")
marcas_sin_datos = set(MARCAS_A_EJECUTAR) - marcas_con_datos
if marcas_sin_datos:
    print(f"Marcas SIN datos: {marcas_sin_datos}")

In [ ]:
df_bruto = pd.DataFrame(todas_las_ofertas)

if df_bruto.empty:
    print("No se han extraído ofertas.")
else:
    df = (
        df_bruto
        .drop_duplicates(subset=["marca", "modelo"], keep="first")
        .reset_index(drop=True)
    )

    CAMPOS_ORDENADOS = [
        "marca", "modelo", "tipo_combustible", "precio_vehiculo",
        "cuota_mensual", "plazo_meses", "entrada", "tin", "tae",
        "comision_apertura", "porcentaje_comision_apertura",
        "valor_residual", "importe_financiado", "tipo_financiacion",
        "fecha_fin_oferta", "url", "fecha_extraccion"
    ]
    cols = [c for c in CAMPOS_ORDENADOS if c in df.columns]
    df = df[cols]

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_rows", 100)

    print(f"Ofertas brutas: {len(df_bruto)} → tras deduplicar: {len(df)}")
    print(f"\nModelos por marca:")
    print(df.groupby("marca")["modelo"].count().to_string())
    display(df)

In [ ]:
# Exportar a CSV
nombre_archivo = f"ofertas_competencia_{date.today().strftime('%Y%m%d')}.csv"
df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
print(f"Guardado en: {nombre_archivo}")

# En Colab, descargar el archivo:
# from google.colab import files
# files.download(nombre_archivo)

In [ ]:
# Resumen comparativo por marca
if not df.empty and "marca" in df.columns:
    resumen = df.groupby("marca").agg(
        num_ofertas=("modelo", "count"),
        tin_medio=("tin", "mean"),
        tae_medio=("tae", "mean"),
        cuota_min=("cuota_mensual", "min"),
        cuota_max=("cuota_mensual", "max")
    ).round(2)
    print(resumen)